# Train (Mac / Apple Silicon)

LoRA fine-tune using **MLX** — Apple's native ML framework. Runs on any M1/M2/M3/M4 Mac. No CUDA, no NVIDIA, no Colab.

**Memory guide:**
- 8 GB Mac → Llama-3.2-1B (4-bit) comfortably
- 16 GB Mac → Llama-3.2-3B (4-bit) comfortably  
- 36 GB+ Mac → Llama-3.1-8B (4-bit) or bigger

**Run this in Jupyter Lab locally** — not in Colab, not in Kaggle.
Launch with `jupyter lab studio/notebooks/train_mac.ipynb` from the repo root.

In [ ]:
# Check you're on Apple Silicon with enough memory.
import platform, subprocess
print(f'machine: {platform.machine()}')
assert platform.machine() == 'arm64', 'This notebook is for Apple Silicon (arm64).'
print(subprocess.check_output(['sysctl', 'hw.memsize']).decode().strip())

In [ ]:
# Install MLX. One-time setup.
!pip install -q mlx-lm

In [ ]:
import os
# Repo root — adjust if you cloned elsewhere.
REPO = os.path.expanduser('~/Downloads/jailbreak_tut_sessions')
os.chdir(REPO)
!pwd

In [ ]:
# Dataset.
# Use studio/data/exports/alpaca.jsonl once you've hit 'Rebuild' on /dataset.
# For a smoke test, the sample file works.
DATASET = 'studio/scripts/_sample_alpaca.jsonl'

import json
with open(DATASET) as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f'rows: {len(rows)}')
print(f'first row keys: {list(rows[0].keys())}')

In [ ]:
# Dry-run — validates dataset shape without loading MLX.
!python studio/scripts/train_mlx.py \
  --dataset $DATASET \
  --output studio/data/finetunes/mac-smoke \
  --dry-run

In [ ]:
# Real training.
# 500 iters on 16 GB Mac + 3B 4-bit model ≈ 10-20 minutes.
# --iters is iterations, not epochs. A rough guide: iters ≈ num_rows × epochs / batch_size.
# For bigger datasets, bump --iters to 1000-2000.
!python studio/scripts/train_mlx.py \
  --model mlx-community/Llama-3.2-3B-Instruct-4bit \
  --dataset $DATASET \
  --output studio/data/finetunes/mac-llama3.2-3b \
  --iters 500 \
  --lora-layers 16 \
  --batch-size 2 \
  --learning-rate 1e-5

In [ ]:
# Sanity test — generate with the LoRA adapter attached.
!python -m mlx_lm.generate \
  --model mlx-community/Llama-3.2-3B-Instruct-4bit \
  --adapter-path studio/data/finetunes/mac-llama3.2-3b/adapters \
  --prompt 'write a one-sentence product blurb for a leather notebook' \
  --max-tokens 120

In [ ]:
# Fuse / merge LoRA into base model. Needed before converting to GGUF for Ollama.
!python studio/scripts/train_mlx.py \
  --model mlx-community/Llama-3.2-3B-Instruct-4bit \
  --dataset $DATASET \
  --output studio/data/finetunes/mac-llama3.2-3b \
  --iters 1 \
  --merge

## Next steps — get it into Ollama

Option A (simpler, slower inference) — serve the MLX model directly:
```bash
python -m mlx_lm.server \
  --model studio/data/finetunes/mac-llama3.2-3b/merged \
  --port 8080
```
Then point `/attack` at it: `provider=__custom__`, `base_url=http://localhost:8080/v1`, `model=any`, `api_key=mlx`.

Option B (faster inference via llama.cpp + Ollama) — convert the merged model to GGUF:
```bash
# Clone llama.cpp and build it (one-time)
git clone https://github.com/ggerganov/llama.cpp ~/llama.cpp
cd ~/llama.cpp && cmake -B build -DGGML_METAL=ON && cmake --build build --config Release -j

# Convert + quantize
export LLAMA_CPP_DIR=~/llama.cpp
python studio/scripts/export_gguf.py \
  --adapter studio/data/finetunes/mac-llama3.2-3b/merged \
  --base studio/data/finetunes/mac-llama3.2-3b/merged \
  --output studio/data/finetunes/mac-llama3.2-3b.gguf \
  --quant q4_k_m

ollama create jb-llama3.2 -f Modelfile
```

Then test in `/attack` → benchmark in `/eval`. Compare the before/after in the eval page.